# Client A notebook

Vicky - data prep. Zac - Client A strategy.

In [2]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## Vicky - Data prep

Gets the stock data, checks which tickers are eligible and converts prices to CAD.

In [3]:
import time  # not in the given import cell; needed for time.sleep

GROUP_NUMBER = "6"        
TICKER_FILE = "Tickers.csv"

PURCHASE_DATE = pd.Timestamp("2026-11-27")
# Competition holding window (used to find earnings reports during the competition)
HOLD_START = pd.Timestamp("2026-11-27 16:00", tz="America/New_York")
HOLD_END   = pd.Timestamp("2026-12-04 16:00", tz="America/New_York")

# Price history to download: must start before the volume window and cover a year of signals
HISTORY_START = pd.Timestamp("2024-10-01")

# Volume rule
VOLUME_START = pd.Timestamp("2025-10-01")
VOLUME_END   = pd.Timestamp("2026-09-30")
MIN_AVG_VOLUME = 5000
MIN_TRADING_DAYS_IN_MONTH = 18

# Client mandates
CLIENT_BUDGETS = {"A": 1_000_000, "B": 750_000}       # CAD
CLIENT_N_STOCKS = {"A": 12, "B": 20}                   # our chosen portfolio sizes
MIN_STOCKS, MAX_STOCKS = 10, 20
MAX_STOCK_WEIGHT = 0.15
MAX_SECTOR_WEIGHT = 0.40

# Transaction fee: the smaller of US$2.15 and US$0.001 per share, for every stock
MAX_FEE_USD = 2.15
FEE_PER_SHARE_USD = 0.001
SAFETY_BUFFER_CAD = 0.01     # leave one cent unspent so rounding can never exceed the budget

# Eligibility (yfinance codes)
ALLOWED_QUOTE_TYPES = ["EQUITY"]
ALLOWED_CURRENCIES  = ["CAD", "USD"]
US_EXCHANGES = ["NMS", "NGM", "NCM", "NYQ", "ASE", "PCX", "BTS"]  # Nasdaq tiers, NYSE, NYSE American, NYSE Arca, Cboe
CA_EXCHANGES = ["TOR", "VAN", "NEO", "CNQ"]                       # TSX, TSX Venture, Cboe Canada, CSE
# OTC / pink-sheet codes are deliberately excluded: they are not exchange listings.
MAX_STALE_DAYS = 5           # last price older than this before the purchase date = not trading

# Strategy parameters
TRADING_DAYS_PER_YEAR = 252
LOOKBACK_DAYS = 252          # one year: Client B volatility and the equal-weight comparison period
MOMENTUM_DAYS = 126          # about six months
SHORT_MOMENTUM_DAYS = 20     # about one month
TREND_DAYS = 50              # 50-day moving average trend filter
CATALYST_BONUS = 0.10        # score bonus for Client A stocks with earnings in the window and positive momentum

BENCHMARK_TICKERS = {"US": "^GSPC", "CA": "^GSPTSE"}  # S&P 500 and S&P/TSX Composite
FX_TICKER = "CAD=X"          # CAD per 1 USD
PAUSE_SECONDS = 0.2          # pause between per-ticker requests to avoid rate limits





## read the ticker file and collect company information

def read_tickers(path):
    """Read the ticker file (one ticker per row, no header) in order to return a clean, de-duplicated list."""
    raw = pd.read_csv(path, header=None) ## pd haven't touched the data
    tickers = (raw.iloc[:, 0].dropna().astype(str)
               .str.strip().str.upper())
    tickers = tickers[tickers != ""].drop_duplicates()
    return list(tickers)


def get_stock_info(tickers):
    """Looking up quote type, exchange, currency and sector for each ticker.
    Tickers that yfinance cannot find will result in empty values (they fail the checks later)."""
    rows = []
    for ticker in tickers:
        try:
            info = yf.Ticker(ticker).info
        except Exception:
            info = {}
        rows.append({
            "Ticker": ticker,
            "SecurityType": info.get("quoteType"),
            "Exchange": info.get("exchange"),
            "Currency": info.get("currency"),
            "Sector": info.get("sector"),
            "Name": info.get("longName") or info.get("shortName"),
        })
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows).set_index("Ticker")


def market_of(exchange):
    """Return 'US', 'CA' or None depending on where the stock is listed."""
    if pd.isna(exchange):
        return None
    if exchange in US_EXCHANGES:
        return "US"
    if exchange in CA_EXCHANGES:
        return "CA"
    return None



## download price and volume history
#### Use auto_adjust=False and Close column


def download_history(tickers, start, end):
    """Download daily closes and volumes
       return into two DFs (dates x tickers)"""

    data = yf.download(tickers, start=start, end=end + pd.Timedelta(days=1),
                       auto_adjust=False, progress=False, group_by="column")

    closes = data["Close"]
    volumes = data["Volume"]
    # With one ticker yfinance can return a Series; make it a DataFrame either way
    if isinstance(closes, pd.Series):
        closes = closes.to_frame(tickers[0])
        volumes = volumes.to_frame(tickers[0])
    closes.index = pd.to_datetime(closes.index).tz_localize(None)
    volumes.index = pd.to_datetime(volumes.index).tz_localize(None)
    return closes, volumes 



def get_fx_rate(index, start, end):
    """CAD per USD, lined up with the stock dates (forward-filled over holidays)."""
    fx = yf.download(FX_TICKER, start=start, end=end + pd.Timedelta(days=1),
                     auto_adjust=False, progress=False)["Close"]
    if isinstance(fx, pd.DataFrame):
        fx = fx.iloc[:, 0]
    fx.index = pd.to_datetime(fx.index).tz_localize(None)
    return fx.reindex(fx.index.union(index)).ffill().reindex(index).rename("USDCAD")



def convert_to_cad(closes_local, currencies, fx):
    """Multiply USD-quoted columns by the daily exchange rate; CAD columns stay as they are."""
    prices_cad = closes_local.copy()
    for ticker in closes_local.columns:
        if currencies.get(ticker) == "USD":
            prices_cad[ticker] = closes_local[ticker] * fx
    return prices_cad




## Volume Rule
## A "trading day" is a day the stock actually has a volume record. Because TSX and US holidays differ, each stock is counted on its own calendar.
## The average is taken over all remaining days (not an average of monthly averages).


def average_volume(volume_series, start, end, min_days):
    """Average daily volume over [start, end], excluding months with fewer than min_days trading days."""
    v = volume_series.loc[start:end].dropna()
    if v.empty:
        return np.nan
    month = v.index.to_period("M")
    days_per_month = v.groupby(month).count()
    full_months = days_per_month[days_per_month >= min_days].index
    v = v[month.isin(full_months)]
    return v.mean() if not v.empty else np.nan


## Eligibility check + load_stock_data()
def check_eligibility(info, closes_local, volumes, as_of):
    """Add AvgVolume, Eligible and Reason columns. Every failed rule is listed in Reason."""
    result = info.copy()
    result["Market"] = result["Exchange"].apply(market_of)
    result["AvgVolume"] = np.nan
    result["Eligible"] = False
    result["Reason"] = ""

    for ticker in result.index:
        problems = []
        row = result.loc[ticker]

        if pd.isna(row["SecurityType"]):
            # yfinance found nothing: report one clear reason and skip the other checks
            result.loc[ticker, "Reason"] = "invalid ticker (no data found)"
            continue
        if row["SecurityType"] not in ALLOWED_QUOTE_TYPES:
            problems.append(f"not a company stock ({row['SecurityType']})")
        if pd.isna(row["Market"]):
            problems.append(f"not listed on a Canadian/US exchange ({row['Exchange']})")
        if row["Currency"] not in ALLOWED_CURRENCIES:
            problems.append(f"currency not CAD/USD ({row['Currency']})")

        if ticker in closes_local.columns:
            prices = closes_local[ticker].dropna()
            last_date = prices.index.max() if not prices.empty else None
            check_date = min(as_of, pd.Timestamp.today().normalize())
            if last_date is None or (check_date - last_date).days > MAX_STALE_DAYS:
                problems.append("no recent price (possibly delisted)")
            avg_vol = average_volume(volumes[ticker], VOLUME_START, VOLUME_END,
                                     MIN_TRADING_DAYS_IN_MONTH)
            result.loc[ticker, "AvgVolume"] = avg_vol
            if pd.isna(avg_vol) or avg_vol < MIN_AVG_VOLUME:
                problems.append(f"average volume below {MIN_AVG_VOLUME:,}")
        else:
            problems.append("no price data")

        result.loc[ticker, "Eligible"] = len(problems) == 0
        result.loc[ticker, "Reason"] = "; ".join(problems)
    return result


def load_stock_data(path, as_of=PURCHASE_DATE):
    """Read tickers, check eligibility and download history.
    Returns a dictionary with the screening table and all historical tables."""
    tickers = read_tickers(path)
    print(f"Read {len(tickers)} unique tickers from {path}")

    info = get_stock_info(tickers)
    found = [t for t in tickers if pd.notna(info.loc[t, "SecurityType"])]
    closes_local, volumes = download_history(found, HISTORY_START, as_of)

    screening = check_eligibility(info, closes_local, volumes, as_of)
    eligible = list(screening.index[screening["Eligible"]])

    # Keep only eligible stocks in the price tables
    closes_local = closes_local[eligible].loc[:as_of]
    volumes = volumes[eligible].loc[:as_of]

    fx = get_fx_rate(closes_local.index, HISTORY_START, as_of)
    prices_cad = convert_to_cad(closes_local, screening["Currency"].to_dict(), fx)

    print(f"{len(eligible)} eligible, {len(tickers) - len(eligible)} excluded")
    return {
        "screening": screening,
        "eligible": eligible,
        "closes_local": closes_local,
        "volumes": volumes,
        "fx": fx,
        "prices_cad": prices_cad,
    }


## benchmarks in CAD

## Each stock is compared with home markets

def get_benchmarks_cad(index, fx, start, end):
    """Return a DataFrame with columns 'US' and 'CA', both in CAD, lined up with the stock dates."""
    symbols = list(BENCHMARK_TICKERS.values())
    closes, _ = download_history(symbols, start, end)
    closes = closes.reindex(closes.index.union(index)).ffill().reindex(index)
    benchmark_cad = pd.DataFrame(index=index)
    benchmark_cad["US"] = closes[BENCHMARK_TICKERS["US"]] * fx   # S&P 500 is in USD
    benchmark_cad["CA"] = closes[BENCHMARK_TICKERS["CA"]]        # TSX is already in CAD
    return benchmark_cad





## Catalysts
def earnings_in_window(timestamps, start=HOLD_START, end=HOLD_END):
    """Return the first earnings timestamp inside the window, or NaT."""
    for ts in sorted(timestamps):
        ts = pd.Timestamp(ts)
        if ts.tzinfo is None:
            ts = ts.tz_localize("America/New_York")
        else:
            ts = ts.tz_convert("America/New_York")
        if ts.hour == 0 and ts.minute == 0:      # time unknown: compare dates only
            if start.date() < ts.date() <= end.date():
                return ts
        elif start < ts < end:
            return ts
    return pd.NaT


def get_earnings_timestamps(ticker):
    """Collect earnings timestamps from get_earnings_dates(), falling back to .calendar."""
    stock = yf.Ticker(ticker)
    timestamps = []
    try:
        dates = stock.get_earnings_dates(limit=8)
        if dates is not None and len(dates) > 0:
            timestamps += list(dates.index)
    except Exception:
        pass
    if not timestamps:
        try:
            calendar = stock.calendar or {}
            timestamps += list(calendar.get("Earnings Date", []))
        except Exception:
            pass
    return timestamps


def get_catalysts(tickers):
    """Return a DataFrame with EarningsDate and CatalystStatus for each ticker."""
    rows = []
    for ticker in tickers:
        timestamps = get_earnings_timestamps(ticker)
        hit = earnings_in_window(timestamps)
        if pd.notna(hit):
            status = "Event"
        elif not timestamps:
            status = "Unknown"
        # Only say "NoEvent" if we can see an upcoming date (otherwise the data may just be stale)
        elif max(pd.Timestamp(t).tz_localize(None) if pd.Timestamp(t).tzinfo is None
                 else pd.Timestamp(t).tz_convert(None) for t in timestamps) > HOLD_START.tz_convert(None):
            status = "NoEvent"
        else:
            status = "Unknown"
        rows.append({"Ticker": ticker, "EarningsDate": hit, "CatalystStatus": status})
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows).set_index("Ticker")



## Volatility

def window_return(prices, days):
    """Return over the last `days` trading days (prices must have no gaps)."""
    if len(prices) <= days:
        return np.nan
    return prices.iloc[-1] / prices.iloc[-1 - days] - 1


def calculate_signals(prices_cad, volumes, benchmark_cad, markets, as_of=PURCHASE_DATE):
    """Return a DataFrame of signals (one row per ticker), calculated as of `as_of`."""
    prices_cad = prices_cad.loc[:as_of]
    volumes = volumes.loc[:as_of]
    benchmark_cad = benchmark_cad.loc[:as_of]

    # Benchmark 20-day returns, using the benchmark's own trading days
    bench_return20 = {m: window_return(benchmark_cad[m].dropna(), 20) for m in benchmark_cad.columns}

    rows = []
    for ticker in prices_cad.columns:
        prices = prices_cad[ticker].dropna()
        daily_returns = prices.pct_change().dropna()
        vol = volumes[ticker].dropna()

        vol20 = daily_returns.tail(20).std() if len(daily_returns) >= 20 else np.nan
        vol60 = daily_returns.tail(60).std() if len(daily_returns) >= 60 else np.nan
        return20 = window_return(prices, 20)
        ma20 = prices.tail(20).mean() if len(prices) >= 20 else np.nan
        avg_vol_60 = vol.tail(60).mean() if len(vol) >= 60 else np.nan

        rows.append({
            "Ticker": ticker,
            "Vol20": vol20,
            "Vol60": vol60,
            "VolRatio": vol20 / vol60 if vol60 else np.nan,
            "Return5": window_return(prices, 5),
            "Return20": return20,
            "PriceVsMA20": prices.iloc[-1] / ma20 - 1 if ma20 else np.nan,
            "RelativeMomentum": return20 - bench_return20.get(markets.get(ticker), np.nan),
            "RelativeVolume": vol.tail(5).mean() / avg_vol_60 if avg_vol_60 else np.nan,
        })
    signals = pd.DataFrame(rows).set_index("Ticker")

    # Percentile ranks (0 to 1) make the signals comparable for Zac's scoring
    for column in ["Vol20", "Return20", "RelativeMomentum", "RelativeVolume"]:
        signals[column + "Rank"] = signals[column].rank(pct=True)
    return signals



## running everything

data = load_stock_data(TICKER_FILE)
screening = data["screening"]

# Show what was excluded and why
exclusions = screening.loc[~screening["Eligible"], ["SecurityType", "Exchange", "Currency", "Reason"]]
print(f"\nExcluded {len(exclusions)} tickers:")
display(exclusions)

Read 40 unique tickers from Tickers.csv


$MON: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2024-10-01 00:00:00 -> 2026-11-28 00:00:00)
$RTN: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found

5 Failed downloads:
['MON', 'RTN', 'AGN', 'CELG']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2024-10-01 00:00:00 -> 2026-11-28 00:00:00)


35 eligible, 5 excluded

Excluded 5 tickers:


,SecurityType,Exchange,Currency,Reason
Ticker,,,,
AGN,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...
BK,EQUITY,NYQ,NaN,currency not CAD/USD (nan); no recent price (p...
CELG,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...
MON,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...
RTN,NONE,NaN,NaN,not a company stock (NONE); not listed on a Ca...


## Client A - Long-term growth

Choose stocks for long-term capital growth using price momentum, company profitability and reasonable valuation. The one-week competition is only a short snapshot of this investment approach.

In [ ]:
# Long-term return and risk indicators, using CAD prices
prices = data["prices_cad"]
VOL_DAYS = 60


def stock_scores(day):
    rows = []
    for ticker in data["eligible"]:
        p = prices[ticker].loc[:day].dropna()
        if len(p) < VOL_DAYS + 1 or day not in p.index:
            continue
        sector = screening.loc[ticker, "Sector"]
        if pd.isna(sector):
            continue

        six_month = np.nan
        one_year = np.nan
        if len(p) > MOMENTUM_DAYS:
            six_month = p.iloc[-1] / p.iloc[-MOMENTUM_DAYS - 1] - 1
        if len(p) > LOOKBACK_DAYS:
            one_year = p.iloc[-1] / p.iloc[-LOOKBACK_DAYS - 1] - 1

        rows.append({
            "Ticker": ticker,
            "Sector": sector,
            "SixMonth": six_month,
            "OneYear": one_year,
            "Trend": p.iloc[-1] / p.tail(TREND_DAYS).mean() - 1,
            "Volatility": p.pct_change().tail(VOL_DAYS).std()
        })

    if len(rows) < MIN_STOCKS:
        raise ValueError("Not enough stocks with usable price history")

    s = pd.DataFrame(rows).set_index("Ticker")
    # Rank returns and trends against the other eligible stocks
    s["OneYearRank"] = s["OneYear"].rank(pct=True).fillna(0.5)
    s["SixMonthRank"] = s["SixMonth"].rank(pct=True).fillna(0.5)
    s["PriceScore"] = (0.30 * s["OneYearRank"]
                       + 0.25 * s["SixMonthRank"]
                       + 0.10 * s["Trend"].rank(pct=True)
                       + 0.05 * (1 - s["Volatility"].rank(pct=True)))

    # Choose a diverse pool before checking fundamentals
    candidates = []
    counts = {}
    ranked = s.sort_values("PriceScore", ascending=False)
    for ticker in ranked.index:
        sector = s.loc[ticker, "Sector"]
        if counts.get(sector, 0) >= int(MAX_STOCKS * MAX_SECTOR_WEIGHT):
            continue
        candidates.append(ticker)
        counts[sector] = counts.get(sector, 0) + 1
        if len(candidates) == MAX_STOCKS:
            break
    s = s.loc[candidates].copy()
    if len(s) < MIN_STOCKS:
        raise ValueError("Not enough candidates from different sectors")

    # Check the same fundamental ratios for every candidate
    s["PE"] = np.nan
    s["PB"] = np.nan
    s["ROE"] = np.nan
    s["LossMaking"] = False
    for ticker in candidates:
        try:
            info = yf.Ticker(ticker).info
        except Exception:
            info = {}
        pe = info.get("trailingPE")
        pb = info.get("priceToBook")
        roe = info.get("returnOnEquity")
        eps = info.get("trailingEps")
        if isinstance(pe, (int, float)) and np.isfinite(pe) and pe > 0:
            s.loc[ticker, "PE"] = pe
        if isinstance(pb, (int, float)) and np.isfinite(pb) and pb > 0:
            s.loc[ticker, "PB"] = pb
        if isinstance(roe, (int, float)) and np.isfinite(roe):
            s.loc[ticker, "ROE"] = roe
        if isinstance(eps, (int, float)) and np.isfinite(eps) and eps <= 0:
            s.loc[ticker, "LossMaking"] = True
        if isinstance(pe, (int, float)) and np.isfinite(pe) and pe <= 0:
            s.loc[ticker, "LossMaking"] = True
        time.sleep(PAUSE_SECONDS)

    # Lower positive P/E and P/B are better; losses get a low P/E score
    s["PEScore"] = s["PE"].rank(pct=True, ascending=False).fillna(0.5)
    s.loc[s["LossMaking"], "PEScore"] = 0
    s["Growth"] = (s["PriceScore"]
                   + 0.15 * s["ROE"].rank(pct=True).fillna(0.5)
                   + 0.10 * s["PEScore"]
                   + 0.05 * s["PB"].rank(pct=True, ascending=False).fillna(0.5))
    return s

In [ ]:
# Select the highest-ranked stocks, keeping sector and stock limits

def select_stocks(s):
    ranked = s.sort_values("Growth", ascending=False)
    sizes = list(range(CLIENT_N_STOCKS["A"], MAX_STOCKS + 1))
    sizes.extend(range(MIN_STOCKS, CLIENT_N_STOCKS["A"]))

    for n in sizes:
        chosen = []
        counts = {}
        for ticker in ranked.index:
            sector = s.loc[ticker, "Sector"]
            if counts.get(sector, 0) >= int(n * MAX_SECTOR_WEIGHT):
                continue
            chosen.append(ticker)
            counts[sector] = counts.get(sector, 0) + 1
            if len(chosen) == n:
                break
        if len(chosen) != n:
            continue

        # Begin with half the budget equally weighted, then favour higher scores
        weights = pd.Series(0.5 / n, index=chosen)
        for ticker in chosen:
            sector = s.loc[ticker, "Sector"]
            same = []
            for t in chosen:
                if s.loc[t, "Sector"] == sector:
                    same.append(t)
            extra = min(1 - weights.sum(), MAX_STOCK_WEIGHT - weights[ticker],
                        MAX_SECTOR_WEIGHT - weights[same].sum())
            if extra > 0:
                weights[ticker] = weights[ticker] + extra
        if np.isclose(weights.sum(), 1):
            return chosen, weights

    raise ValueError("Could not build a valid Client A portfolio")

In [ ]:
# Construct the long-term growth portfolio using the purchase-day closing prices
# The TA must run this after November 27 closing prices are available
day = PURCHASE_DATE
if day not in prices.index:
    raise ValueError("November 27 closing prices are not available yet")

s = stock_scores(day)
chosen, weights = select_stocks(s)
cad = prices.loc[day, chosen]
local = data["closes_local"].loc[day, chosen]
fx = float(data["fx"].loc[day])
if cad.isna().any() or local.isna().any() or pd.isna(fx) or fx <= 0:
    raise ValueError("Missing closing prices or exchange rate")

budget = CLIENT_BUDGETS["A"]
money = budget - SAFETY_BUFFER_CAD

# Fees depend on shares, and shares depend on the money left after fees
for i in range(5):
    shares = money * weights / cad
    fees = np.minimum(MAX_FEE_USD, FEE_PER_SHARE_USD * shares).sum() * fx
    money = budget - fees - SAFETY_BUFFER_CAD

shares = money * weights / cad
fees = np.minimum(MAX_FEE_USD, FEE_PER_SHARE_USD * shares).sum() * fx
Portfolio_Final_A = pd.DataFrame({
    "Ticker": chosen,
    "Price": local.values,
    "Currency": screening.loc[chosen, "Currency"].values,
    "Shares": shares.values,
    "Value": (shares * cad).values,
    "Weight": weights.values
}, index=range(1, len(chosen) + 1))

# Check the portfolio before displaying it
sector_weights = weights.groupby(screening.loc[chosen, "Sector"]).sum()
assert MIN_STOCKS <= len(chosen) <= MAX_STOCKS
assert set(chosen).issubset(data["eligible"])
assert weights.max() <= MAX_STOCK_WEIGHT + 0.0000001
assert sector_weights.max() <= MAX_SECTOR_WEIGHT + 0.0000001
assert np.isclose(weights.sum(), 1)
assert Portfolio_Final_A["Value"].sum() + fees <= budget

print("Purchase date:", day.date())
display(Portfolio_Final_A)
print("Fees:", round(fees, 2))
print("Cash left:", round(budget - Portfolio_Final_A["Value"].sum() - fees, 2))

## Client A: financial explanation

**Long-term goal.** Client A wants to achieve the highest expected growth on their portfolio, not the lowest risk. So we built an aggressive equity portfolio that's intended to be held for several years. The November 27th through December 4th competition only measures one week of that plan. We do not choose companies by trying to predict the next five trading days, but rather companies with strong long-term growth potential that are trading at reasonable valuations.

**Why this strategy?** We use *growth at a reasonable price*. Strong companies can continue
to grow, but high past returns by themselves do not make a stock a good investment.
Our score uses 30% one-year momentum, 25% six-month momentum and 10% the 50-day price
trend to look for sustained price strength. We add 15% return on equity (ROE) to favour
profitable businesses, 10% lower positive P/E and 5% lower positive price-to-book (P/B)
to avoid blindly paying high valuations, and 5% lower recent volatility for some risk
control. We put most weight on sustained price growth because Client A seeks high returns,
with smaller weights on value, profitability and risk. These are chosen investment
rules, not proven optimal weights. We also test whether moving five percentage
points from one-year to six-month momentum changes our stock picks.

**Connection to Benjamin Graham.** Graham's main lesson for this approach is to pay
attention to the *price of the business*, rather than buy a stock just because its price
is rising. P/E and P/B give simple valuation checks while ROE adds a quality check.
This is not a full calculation of intrinsic value or Graham's margin of safety. We do
not estimate discounted cash flows. We choose up to 20 candidates using price strength and sector variety,
then check the same ratios for every candidate. A company with non-positive
earnings gets the lowest P/E score. Missing data receives a neutral score.

**Portfolio construction.** We prefer 12 stocks, within the required 10–20. Each stock
starts at half an equal weight, then the highest-ranked stocks receive more investment
without exceeding 15% per stock or 40% per sector. Both Canadian and US investments
are valued in CAD, with transaction fees paid from the $1 million budget. In a real
long-term account, we would review the investments about every six months, not trade
based on the result of one week.

**What supports it, and what might go wrong?** Below we compare the selected companies
with other eligible stocks and compare our portfolio with an equal-weight buy-and-hold
portfolio of the *same stocks*. We show one year and, where possible, up to two years
of historical returns, volatility and drawdown, plus market benchmarks. These results
are descriptive, not proof of future outperformance: we chose the stocks at the end
of the historical period, so hindsight and survivorship bias remain. Momentum can
reverse, valuations vary across sectors, ROE can be increased by debt, and our price
returns do not include dividends. The strategy seeks high long-run expected returns,
but no ranking system can guarantee the highest possible return.

In [ ]:
# Show the reasons behind the selected portfolio
weights_A = Portfolio_Final_A.set_index("Ticker")["Weight"]
sectors_A = screening.loc[weights_A.index, "Sector"]
markets_A = screening.loc[weights_A.index, "Market"]

print("Sector weights (%):")
print((weights_A.groupby(sectors_A).sum() * 100).round(1).to_string())
print("US versus Canada (%):")
print((weights_A.groupby(markets_A).sum() * 100).round(1).to_string())

price_cols = ["OneYear", "SixMonth", "Trend", "Volatility"]
price_compare = pd.DataFrame({
    "Chosen stocks": s.loc[weights_A.index, price_cols].median(),
    "Candidates": s[price_cols].median()
})
print("Historical price indicators (median, %):")
print((price_compare * 100).round(2).to_string())

print("Fundamental ratios for chosen stocks (median):")
print(s.loc[weights_A.index, ["PE", "PB", "ROE"]].median().round(2).to_string())
print("Fundamental data checked for", len(s), "candidates")
print("Candidates with non-positive earnings:", int(s["LossMaking"].sum()))

# Check whether a small change in momentum weights changes our selections
alternate = s.copy()
alternate["Growth"] = (s["Growth"] - 0.05 * s["OneYearRank"]
                       + 0.05 * s["SixMonthRank"])
other_stocks, other_weights = select_stocks(alternate)
common = len(set(chosen).intersection(other_stocks))
print("Same selections after changing momentum weights:", common, "out of", len(chosen))

## Client A: historical comparison

The required comparison uses **the same selected stocks**, once with our growth weights and once with equal starting weights. Both are buy-and-hold portfolios in CAD; there is no daily rebalancing. We show up to one year and up to two years of available shared price history, plus the S&P 500 in CAD and TSX price benchmarks. These are retrospective price-only comparisons, not forecasts of the next five years.

In [ ]:
# Compare the same holdings at our weights and at equal weights
benchmarks = get_benchmarks_cad(prices.index, data["fx"], HISTORY_START, day)
tables = []
periods = []
chart = None

for days, label in [(LOOKBACK_DAYS, "Up to 1 year"), (LOOKBACK_DAYS * 2, "Up to 2 years")]:
    window = prices.loc[:day, chosen].tail(days + 1).ffill().dropna()
    if len(window) < 60:
        continue
    if label == "Up to 2 years" and len(window) <= LOOKBACK_DAYS:
        continue

    relative = window / window.iloc[0]
    growth = pd.DataFrame(index=window.index)
    growth["Client A"] = relative.mul(weights, axis=1).sum(axis=1)
    growth["Equal weight"] = relative.mean(axis=1)

    for market, name in [("US", "S&P 500 CAD"), ("CA", "TSX")]:
        b = benchmarks[market].reindex(window.index).ffill()
        if b.notna().all() and b.iloc[0] > 0:
            growth[name] = b / b.iloc[0]

    daily = growth.pct_change().dropna()
    summary = pd.DataFrame(index=growth.columns)
    summary["Return (%)"] = (growth.iloc[-1] - 1) * 100
    summary["Annual volatility (%)"] = daily.std() * np.sqrt(TRADING_DAYS_PER_YEAR) * 100
    summary["Worst drop (%)"] = (growth / growth.cummax() - 1).min() * 100
    summary["Start"] = window.index[0].date()
    summary["End"] = window.index[-1].date()
    summary["Days"] = len(daily)
    summary["Period"] = label
    tables.append(summary)
    periods.append(label)
    chart = growth

if not tables:
    raise ValueError("Not enough common history for an equal-weight comparison")

comparison_table = pd.concat(tables)
display(comparison_table.round(2))
for label in periods:
    part = comparison_table.loc[comparison_table["Period"] == label]
    a = part.loc["Client A", "Return (%)"]
    b = part.loc["Equal weight", "Return (%)"]
    print(label, "growth weights versus equal weights:", round(a - b, 2), "percentage points")

(chart * 100).plot(figsize=(10, 5), title="Buy-and-hold growth of $100 CAD")
plt.ylabel("Value (CAD)")
plt.grid(alpha=0.3)
plt.show()

## Shared validation and CSV outputs

`validate_portfolio()` checks any client's final portfolio against the competition rules and prints a pass/fail
table. `save_portfolio()` writes the CSV files. Both work for Client B too: call them with
`Portfolio_Final_B` and `"B"`.

In [9]:
def validate_portfolio(portfolio, client, fees_cad):
    """Check a final portfolio against every rule. Returns a DataFrame of checks and prints a warning if any fail."""
    
    budget = CLIENT_BUDGETS[client]
    p = portfolio.set_index("Ticker")
    day = prices.index[-1]
    fx_day = float(data["fx"].loc[day])
    sector_w = p["Weight"].groupby(screening.loc[p.index, "Sector"]).sum()
    expected_value = p["Shares"] * p["Price"] * np.where(p["Currency"] == "USD", fx_day, 1.0)
    expected_fees = np.minimum(MAX_FEE_USD, FEE_PER_SHARE_USD * p["Shares"]).sum() * fx_day
    spent = p["Value"].sum() + fees_cad

    
    checks = [
        ("No missing values", not portfolio.isna().any().any()),
        ("No duplicate tickers", p.index.is_unique),
        ("All tickers eligible", set(p.index).issubset(data["eligible"])),
        (f"Between {MIN_STOCKS} and {MAX_STOCKS} stocks ({len(p)})", MIN_STOCKS <= len(p) <= MAX_STOCKS),
        ("All shares positive", (p["Shares"] > 0).all()),
        ("Weights add to 100%", np.isclose(p["Weight"].sum(), 1)),
        (f"Largest stock weight {p['Weight'].max():.2%} <= {MAX_STOCK_WEIGHT:.0%}",
         p["Weight"].max() <= MAX_STOCK_WEIGHT + 1e-9),
        (f"Largest sector weight {sector_w.max():.2%} <= {MAX_SECTOR_WEIGHT:.0%}",
         sector_w.max() <= MAX_SECTOR_WEIGHT + 1e-9),
        ("Value = Shares x Price x FX", np.allclose(p["Value"], expected_value, rtol=1e-6)),
        ("Weight = Value / total value", np.allclose(p["Weight"], p["Value"] / p["Value"].sum(), atol=1e-6)),
        (f"Fees match the fee rule (CAD {fees_cad:,.2f})", np.isclose(fees_cad, expected_fees, atol=0.01)),
        (f"Value + fees (CAD {spent:,.2f}) <= budget (CAD {budget:,.0f})", spent <= budget),
        (f"Budget used except CAD {budget - spent:,.2f}", budget - spent < 1),
        (f"Prices are from the purchase date ({PURCHASE_DATE.date()})", day == PURCHASE_DATE),
    ]
    
    
    result = pd.DataFrame(checks, columns=["Check", "Passed"])
    print(f"Client {client} validation:")
    print(result.to_string(index=False))
    failed = result.loc[~result["Passed"], "Check"].tolist()
    if failed:
        print("\nWARNING: failed checks:", failed)
    return result


## saving

def save_portfolio(portfolio, client):
    """Write the full portfolio table and a Ticker/Shares file for submission."""
    full_name = f"Portfolio_Final_{client}_Group_{GROUP_NUMBER}.csv"
    stocks_name = f"Stocks_Final_{client}_Group_{GROUP_NUMBER}.csv"
    portfolio.to_csv(full_name, index=False)
    portfolio[["Ticker", "Shares"]].to_csv(stocks_name, index=False)
    print("Saved", full_name, "and", stocks_name)


validation_A = validate_portfolio(Portfolio_Final_A, "A", fees)
save_portfolio(Portfolio_Final_A, "A")

# Shared files used by both clients
screening.to_csv(f"Screening_Group_{6}.csv")
comparison_table.to_csv(f"Comparison_A_Group_{6}.csv")
print(f"Saved Screening_Group_{6}.csv and Comparison_A_Group_{6}.csv")

Client A validation:
                                                  Check  Passed
                                      No missing values    True
                                   No duplicate tickers    True
                                   All tickers eligible    True
                          Between 10 and 20 stocks (12)    True
                                    All shares positive    True
                                    Weights add to 100%    True
                     Largest stock weight 15.00% <= 15%    True
                    Largest sector weight 40.00% <= 40%    True
                            Value = Shares x Price x FX    True
                           Weight = Value / total value    True
                     Fees match the fee rule (CAD 8.87)    True
Value + fees (CAD 999,999.99) <= budget (CAD 1,000,000)    True
                            Budget used except CAD 0.01    True
         Prices are from the purchase date (2026-11-27)   False

Saved Portfolio_Fi